# Data aggregation for 2026 matches

In [8]:
import matplotlib.pyplot as plt
import pandas as pd

Load data

In [9]:
predictions = pd.read_csv("predictions/predictions_2026_group_matches.csv")
predictions.head()

knockout_matches = pd.read_csv("matches/2026_knockout_matches.csv")

Analyse data

In [10]:
predictions = predictions.sort_values(
    by=["group", "confidence"],
    ascending=[True, False]
)
wins = predictions[["group", "predicted_winner"]].copy()

wins["wins"] = 1

group_wins = (
    wins
    .groupby(["group", "predicted_winner"])
    .sum()
    .reset_index()
    .rename(columns={"predicted_winner": "team"})
)

group_wins["rank"] = (
    group_wins
    .groupby("group")["wins"]
    .rank(method="first", ascending=False)
).astype(int)

group_wins = group_wins.sort_values(
    by=["group", "rank"],
    ascending=[True, True]
)

group_wins = group_wins[group_wins["rank"] <= 3]

Output group win predictions as CSV

In [11]:
print(group_wins)
group_wins.to_csv(
    "predictions/predictions_2026_group_wins.csv",
    index=False
)

   group                    team  wins  rank
2      A             South Korea     3     1
1      A            South Africa     2     2
0      A          Czech Republic     1     3
3      B  Bosnia and Herzegovina     3     1
5      B             Switzerland     2     2
4      B                   Qatar     1     3
6      C                  Brazil     3     1
8      C                Scotland     2     2
7      C                   Haiti     1     3
9      D               Australia     2     1
10     D                Paraguay     2     2
11     D                  Turkey     2     3
14     E                 Germany     3     1
12     E                 Curacao     1     2
13     E                 Ecuador     1     3
17     F             Netherlands     3     1
18     F                  Sweden     2     2
16     F                   Japan     1     3
19     G                 Belgium     3     1
21     G                    Iran     2     2
20     G                   Egypt     1     3
23     H  

Fill knockout matches with group winners

In [12]:
winners = (
    group_wins[group_wins["rank"] == 1]
    .set_index("group")["team"]
    .to_dict()
)

runners_up = (
    group_wins[group_wins["rank"] == 2]
    .set_index("group")["team"]
    .to_dict()
)

third_place = group_wins[group_wins["rank"] == 3][["group", "team"]]

team_confidence = pd.concat([
    predictions[["group", "team1", "confidence"]]
        .rename(columns={"team1": "team"}),

    predictions[["group", "team2", "confidence"]]
        .rename(columns={"team2": "team"})
])

third_place_strength = (
    third_place
    .merge(team_confidence, on=["group", "team"])
    .groupby(["group", "team"])["confidence"]
    .mean()
    .reset_index(name="avg_confidence")
    .sort_values(by="avg_confidence", ascending=False)
)

third_place_strength.head(8)

,group,team,avg_confidence
5,F,Japan,0.858583
2,C,Haiti,0.816000
11,L,England,0.815083
6,G,Egypt,0.789722
7,H,Uruguay,0.746968
1,B,Qatar,0.741139
4,E,Ecuador,0.714560
10,K,Usbekistan,0.704778


In [13]:
def resolve_team(value):
    if value.startswith("WINNER_"):
        group = value.split("_")[1]
        return winners.get(group, value)

    if value.startswith("RUNNER_UP_"):
        group = value.split("_")[2]
        return runners_up.get(group, value)

    if value.startswith("THIRD_"):
        idx = int(value.split("_")[1]) - 1
        return third_place_strength.loc[idx, "team"]

    return value


knockout_matches.loc[:, "team1"] = knockout_matches["team1"].apply(resolve_team)
knockout_matches.loc[:, "team2"] = knockout_matches["team2"].apply(resolve_team)

Output team predictions of knockout matches

In [14]:
knockout_matches.to_csv(
    "predictions/predictions_2026_knockout_matches.csv",
    index=False
)

knockout_matches.head(16)

,match_id,team1,team2,date,round
0,M-2026-73,South Africa,Switzerland,29.6.2026,16 Final
1,M-2026-74,Germany,Czech Republic,30.6.2026,16 Final
2,M-2026-75,Netherlands,Scotland,30.6.2026,16 Final
3,M-2026-76,Brazil,Sweden,30.6.2026,16 Final
4,M-2026-77,Senegal,Qatar,1.7.2026,16 Final
5,M-2026-78,Curacao,France,1.7.2026,16 Final
6,M-2026-79,South Korea,Haiti,1.7.2026,16 Final
7,M-2026-80,Ghana,Turkey,1.7.2026,16 Final
8,M-2026-81,Australia,Ecuador,2.7.2026,16 Final
9,M-2026-82,Belgium,Japan,2.7.2026,16 Final
